<font size="+4">`Signal and Audio Processing`</font>

<font size="+3">`Seminar 09: Intro to Generative Text-to-Speech`</font>

<font size="+2">`Maks Nakhodnov & Dmitry Kropotov`</font>

<font size="+2">`Bremen, 2026`</font>

Что вы узнаете из этого ноутбука:

* **Постановка задачи TTS:** Проблема *One-to-Many*. Оценка генерации речи.

# `1. Постановка задачи TTS и Проблема One-to-Many`

**Text-to-Speech (TTS)** — задача генерации человеческой речи из текста. Если ASR — это задача *распознавания* (сжатия непрерывного зашумленного сигнала в дискретные смысловые токены), то TTS — это задача **генерации**, требующая восстановления утерянной информации.

Пусть $Y = (y_1, y_2, \dots, y_N)$ — входная текстовая последовательность, где $y_n \in V$ (буквы, фонемы или токены). 
Задача TTS — сгенерировать аудиосигнал $S = (\mathbf{s}_1, \dots, \mathbf{s}_T)$ или последовательность акустических признаков $X = (\mathbf{x}_1, \dots, \mathbf{x}_M)$, максимизируя вероятность:

$$ X^* = \arg\max_{X} P(X \mid Y, \mathcal{C}) $$

Где $\mathcal{C}$ — контекстное условие (личность спикера, эмоция, скорость, акустика помещения).

## `1.1. The One-to-Many Problem`

Главная фундаментальная сложность TTS заключается в том, что отображение текста в звук **не инъективно**. Одну и ту же фразу «Привет, как дела?» можно произнести бесконечным числом способов:
* **Pitch, Timbre:** С разным тембром (мужчина, женщина, ребенок).
* **Prosody, Speed:** С разной скоростью и паузами.
* **Energy, Emotions:** С разной интонацией (сарказм, радость, грусть).
* **Environment:** В разных окружениях, с разным фоновым шумом.

<b style='color:red;'>Почему в TTS нельзя просто считать L1/MSE Loss между предсказанной и эталонной спектрограммой, как это делается, например, при генерации картинок?</b>

<details><summary>Ответ:</summary>>> 
    
1. Проблема выравнивания во времени: Речь, сгенерированная чуть быстрее или медленнее эталона, даст большой MSE-лосс, даже если звучит идеально для человека. <br>
2. MSE штрафует модель за выбор другой, но валидной интонации. Модель обучается предсказывать математическое ожидание всех спектрограмм для данного текста, что физически выглядит как размытая спектрограмма без четких гармоник. Такой эффект *Oversmoothing* приводит к генерации скучного, роботизированного голоса.
</details>

## `1.2. Структура TTS пайплайна`

<img src="ASR_vs_TTS.svg" alt="" >

Существует множество вариантов, как выбрать каждую из компонент в TTS пайплайне:

1. **Pre-processing:**
    * Текст $\to$ Нормализация $\to$ Фонемы $\to$ Извлечение просодии.
    * Текст $\to$ BPE-токены \& Аудио-референс $\to$ Дискретные аудио-токены.
    * ...
2. **Acoustic Model:**
    * Фонемы $\to$ Mel-спектрограмма (модели *Tacotron 2*, *FastSpeech 2*).
    * Совместное моделирование семантики и акустики (LLM или Flow-модели).
    * ...
3. **Acoustic Features:**
    * Mel/Linear-спектрограмма
    * Латентные скрытые представления
    * Discrete audio codes
    * ...
5. **Vocoder:**
    * Mel-спектрограмма $\to$ Waveform (модели *HiFi-GAN*, *Vocos*).
    * Декодирование токенов или латентов обратно в сырой звук.
    * ...

Каким бы ни было промежуточное представление (спектрограмма или токены), финальным шагом всегда является конвертация в сырую звуковую волну — одномерный массив амплитуд с частотой $24\,000 - 48\,000$ отсчетов в секунду. Этим занимается **Vocoder**.

<b style='color:red;'>Почему нельзя отказаться от вокодера и заставить акустическую модель предсказывать Raw Waveform напрямую (например, авторегрессионно)?</b>

<details><summary>Ответ:</summary>>> 
Из-за частоты дискретизации. Для 1 секунды аудио с частотой 24 кГц модели нужно сделать 24000 авторегрессионных шагов. WaveNet делала именно так, но её инференс занимал минуты для секундного фрагмента. Вокодеры решают проблему "разрешения": акустическая модель работает на низкой частоте (\~50 Гц), предсказывая смысл и мелодию, а вокодер параллельно генерирует высокочастотные акустические детали до 24 кГц.
</details>

## `1.3. Text Frontend: Нормализация и G2P`

Прежде чем акустическая модель начнет генерировать звук, сырой текст необходимо подготовить. Несмотря на то, что современные LLM-based подходы могут использовать информацию о структуре языка для E2E препроцессинга текста, качественная предобработка всё ещё важна для реальных систем.

### `Text Normalization`

Модели TTS не умеют читать цифры, символы и аббревиатуры напрямую. Нормализация переводит небуквенные символы в их словесное (орфографическое) представление:

| Сырой текст | Контекст | Нормализованный текст (как нужно прочитать) |
| :--- | :--- | :--- |
| `1984` | В **1984** году вышел роман... | *в тысяча девятьсот восемьдесят четвертом* |
| `1984` | Мой пин-код: **1984** | *один девять восемь четыре* |
| `St.` | **St.** Patrick's Day | *Saint* |
| `St.` | Wall **St.** | *Street* |
| `$120.5` | Цена: **$120.5** | *сто двадцать долларов пятьдесят центов* |

**Важно отметить, что эта задача нетривиальна, так как результат разрешения неоднозначностей зависит от контекста!.**

Исторически для решения этой задачи использовались своды правил, написанные лингвистами вручную. Сегодня применяются нейронные модели или LLM, работающие в режиме перевода сырого текста в нормализованный.

<b style='color:red;'>В чём опасность использования нейросетей для текстовой нормализации?</b>

<details><summary>Ответ:</summary>>> 
    
<b>Галлюцинации и пропуск слов.</b> Если ручные правила не знают слово, оно просто будет прочитано по буквам. LLM же может перефразировать текст, выбросить "неудобную" цифру или заменить её на случайную. Поэтому в индустрии до сих пор преобладают гибридные Rule-based + Neural подходы.

</details>

### `Grapheme-to-Phoneme (G2P)`

Буквы (графемы) не всегда однозначно отражают звучание слова. В английском языке одно и то же буквосочетание может читаться десятками способов (e.g., *read* $\to$ /riːd/ или /rɛd/). В русском языке есть редукция гласных (корова $\to$[к а р о́ в а]) и **омографы** (слова, которые пишутся одинаково, но звучат по-разному из-за ударения):

* Я открыл дверной **замóк**.
* Мы поехали на экскурсию в старый **зáмок**.

G2P-модели расставляют ударения и переводят нормализованный текст в транскрипцию.

<b style='color:red;'>Зачем вообще переводить текст в фонемы, если современные Audio LLM могут напрямую работать с BPE-токенами текста?</b>

<details><summary>Ответ:</summary>>> 
    
Использование BPE действительно упрощает пайплайн, и модели учатся неявно связывать токены со звуками. Однако:
1. BPE разбивает слова логически, а не фонетически. Редкие слова, имена собственные или новые заимствования модель прочитает с ошибкой ударения или произношения.
2. Фонемы делают модель более стабильной и позволяют вручную исправлять ошибки произношения в словарях без переобучения всей акустической модели. Тем не менее, SOTA модели все больше склоняются к отказу от явного G2P при масштабировании данных.

</details>

Больше дискуссии по этому вопросу [здесь](https://huggingface.co/blog/hexgrad/g2p).

## `1.4. Методы оценивания и метрики`

### `Метрики производительности`

Оценка скорости TTS пайплайна следует аналогичному пути, что и для ASR:

1. **RTF (Real-Time Factor):**
  
   Это отношение времени, затраченного на обработку аудио, к длительности самого аудио:

   $$\text{RTF} = \frac{\text{Processing Time}}{\text{Audio Duration}}$$
    У современных Flow Matching моделей $\text{RTF} \approx 0.05$.

    Аналогично ASR иногда рассматривают обратную величину: $\text{RTFx} = \frac{1}{\text{RTF}}$

2. **First Chunk Latency:**

   Также как для Streaming ASR было важно время до первого токена, для TTS измеряют время до первого сгенерированного чанка.

<img src="https://pic3.zhimg.com/v2-29920fb92ba21fdc5fcdb6270e1e141c_1440w.jpg" alt="" >

### `Объективные метрики качества`

В задаче генерации нет единой математической метрики качества, так как «натуральность» субъективна. Исторически использовались акустические метрики, но сегодня индустрия перешла к гибридным и LLM-based подходам.

1. **Word Error Rate / CER:**

   Измеряет *Разборчивость* и стабильность. Сгенерированное аудио распознаётся через ASR пайплайн с помощью сильных моделей, обычно *Whisper*. Если генеративная модель "проглатывает" буквы, зацикливается или галлюцинирует, то WER вырастет.

<b style='color:red;'>Почему такой подход будет работать? Разве ASR не добавит своих ошибок распознавания?</b>

<details><summary>Ответ:</summary>>> 

Обычно TTS генерация — генерация аудио студийного качества. Для домена высококачественных аудио без постороннего шума ошибка современных ASR моделей минимальна.
 
</details>
   
2. **Speaker Similarity (SS):**

   Измеряет качество *Zero-Shot Voice Cloning*. Эмбеддинги спикера из оригинального референсного аудио и из сгенерированного аудио извлекаются с помощью предобученной модели. 
   $$ \text{SIM} = \cos(\mathbf{e}_{\text{ref}}, \mathbf{e}_{\text{gen}}) = \frac{\mathbf{e}_{\text{ref}} \cdot \mathbf{e}_{\text{gen}}}{\|\mathbf{e}_{\text{ref}}\| \|\mathbf{e}_{\text{gen}}\|} $$
   Значения $\text{SIM} > 0.7$ обычно означают, что человек не отличит голоса.

| Model | Text token | Speech Token | WER (%) | #INS+DEL | #SUB | SS |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| Original | - | - | 3.01 | 66 | 200 | 69.67 |
| VALL-E (Wang et al., 2023) | Phone | Encodec | 18.70 | 342 | 1312 | 53.19 |
| UniAudio (Yang et al., 2023) | Phone | Encodec | 8.74 | 254 | 519 | 47.56 |
| SpearTTS (Kharitonov et al., 2023) | Phone | Hubert | 6.14 | 133 | 410 | 51.71 |
| Exp-1-LibriTTS | Phone | Hubert | 7.41 | 325 | 409 | 67.85 |
| Exp-2-LibriTTS | Phone | S<sup>3</sup><sub>en</sub> | 5.05 | 122 | 325 | 67.85 |
| Exp-3-LibriTTS | BPE<sub>en</sub> | S<sup>3</sup><sub>en</sub> | 3.93 | 108 | 239 | 67.85 |
| Exp-4-LibriTTS | BPE | S<sup>3</sup> | 4.76 | 134 | 287 | 65.94 |
| Exp-4-Large-scale | BPE | S<sup>3</sup> | **3.17** | **96** | **184** | **69.49** |

3. [**Fréchet DeepSpeech Distance (FDSD) / Kernel DeepSpeech Distance (KDSD):**](https://arxiv.org/pdf/1909.11646)

   Аналог FID/KID в CV. Измеряет дистанцию между распределениями реальной речи и сгенерированной в пространстве признаков предобученной ASR.
   $$ \text{FDSD} = \mathcal{W}_{2}(\mathcal{P}_{\text{reference}}, \mathcal{P}_{\text{source}}) \approx \mathcal{W}_{2}(\mathcal{N}(\mu_r, \Sigma_r), \mathcal{N}(\mu_s, \Sigma_s))$$

   $$
    \text{KDSD} = \text{MMD}(f_{\text{reference}}, f_{\text{source}})^{2}
   $$

### `Субъективные метрики качества`

1. **MOS (Mean Opinion Score):** Оценка людьми от 1 до 5 (обычно, по параметрам Naturalness и Speaker Similarity). Долго и дорого.

| Model | MOS | FDSD | cFDSD | KDSD ×10<sup>5</sup> | cKDSD ×10<sup>5</sup> |
| :--- | :--- | :--- | :--- | :--- | :--- |
| *natural speech* | 4.55 ± 0.075 | 0.161 | N/A | 0 | 0 |
| *WaveNet*, van den Oord et al. (2016) | 4.41 ± 0.069 | | | | |
| *Parallel WaveNet*, van den Oord et al. (2018) | 4.41 ± 0.078 | | | | |
| FullD | 1.889 ± 0.057 | 4.51 | 4.46 | 785 | 782 |
| cRWD<sub>1</sub> | 3.394 ± 0.058 | 0.362 | 0.247 | 35.2 | 30.9 |
| cRWD<sub>{1,2,4,8,15}</sub> | 3.498 ± 0.059 | 0.398 | 0.284 | 42.1 | 37.9 |
| cRWD<sub>1</sub> + uRWD<sub>1</sub> | 3.502 ± 0.057 | 0.259 | 0.144 | 16.6 | 12.3 |
| (cRWD<sub>1</sub> + uRWD<sub>1</sub>)<sup>×5</sup> | 3.526 ± 0.054 | 0.194 | 0.073 | 5.59 | 1.34 |
| RWD<sub>1,240×{1,2,4,8,15}</sub> | 4.154 ± 0.050 | 0.184 | 0.061 | 3.73 | 0.54 |
| RWD<sup>*</sup><sub>480</sub> | 4.195 ± 0.045 | 0.193 | 0.069 | 5.28 | 0.98 |
| GAN-TTS (RWD<sup>*</sup>) | 4.213 ± 0.046 | 0.184 | 0.060 | 3.84 | 0.37 |

2. **ABX Test:** Слушателю дают аудио A (модель 1), аудио B (модель 2) и референс X. Нужно выбрать, что ближе к X.
3. [**MLLM-as-a-Judge**:](https://arxiv.org/pdf/2510.14664v1)

        С ростом популярности *Instruction-guided TTS* (когда модель использует сложные инструкции, например: «прочитай это саркастично»), классические метрики перестали работать. Сейчас стандартом является использование мультимодальных LLM в качестве автоматических асессоров.
    В MLLM загружают сгенерированное аудио, текст и промпт-инструкцию, после чего она оценивает:
    * **Instruction Following:** Выполнила ли модель сложную инструкцию (паузы, акцент, вздохи)?
    * **Expressiveness:** Эмоциональное богатство речи.

### `Датасеты и SOTA Бенчмарки`

В ASR для качественного и робастного распознавания аудио данные должны быть разнообразны. Однако для TTS картина обратная — если в аудио генерации будет произнесено не то, что подаётся на вход модели, TTS выучит неправильные произношения. Более того, unconditional модели без дополнительной информации о говорящем будут усреднять речь разных спикеров, что будет приводить к неестественной генерации. Поэтому для старых TTS моделей почти всегда использовались **чистые single speaker студийные датасеты**. Однако, современные модели, допускающие **условие по говорящему** (speaker embedding, style-токены, семантические подсказки и т.п.), эффективно обучаются на **многоголосых** наборах. Такие модели обучают на сотнях тысяч часов речи (из подкастов, YouTube), чтобы модель видела всё многообразие тембров и акустических условий.

<b style='color:red;'>Датасеты для ASR почти всегда используют частоту дискретизации 16 kHz. Почему для современных TTS датасетов стандартом является 24 kHz или даже 48 kHz?</b>

<details><summary>Ответ:</summary>>> 

1. В ASR главная цель — извлечь смысл. Вся основная лингвистическая и семантическая информация человеческой речи (форманты гласных и большинства согласных) лежит в диапазоне частот до 8 кГц. Согласно теореме Котельникова, частоты дискретизации 16 кГц достаточно, чтобы без потерь восстановить частоты до 8 кГц. 
2. В TTS цель — естественность и качество звучания. Частоты выше 8 кГц содержат высшие обертоны, дыхание, четкие шипящие звуки и индивидуальный окрас тембра. Если сгенерировать голос на 16 кГц, он будет звучать глухо. Для студийного чистого звучания требуется генерация высоких частот, поэтому TTS-датасеты и вокодеры работают на 24 кГц (покрывает частоты до 12 кГц) и выше.

</details>

<table style="width: 100%; border-collapse: collapse; font-family: sans-serif; text-align: left;">
  <tr style="background-color: #f2f2f2; font-weight: bold; border-bottom: 2px solid black;">
    <th style="padding: 10px;">Название датасета</th>
    <th style="padding: 10px;">Размер (Часы)</th>
    <th style="padding: 10px;">Домен / Особенности</th>
    <th style="padding: 10px;">Год</th>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>LJSpeech</b></td>
    <td style="padding: 10px;">24</td>
    <td style="padding: 10px;">Студийная запись, 1 спикер. Исторический baseline для Tacotron/FastSpeech.</td>
    <td style="padding: 10px;">2017</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>LibriTTS</b></td>
    <td style="padding: 10px;">585</td>
    <td style="padding: 10px;">Аудиокниги, 2456 спикеров. Стандарт для ранних multi-speaker моделей.</td>
    <td style="padding: 10px;">2019</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>GigaSpeech</b></td>
    <td style="padding: 10px;">10,000</td>
    <td style="padding: 10px;">Подкасты, YouTube. Спонтанная речь, шумы. Дал старт zero-shot моделям.</td>
    <td style="padding: 10px;">2021</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd;">
    <td style="padding: 10px;"><b>Emilia</b></td>
    <td style="padding: 10px;">> 200,000</td>
    <td style="padding: 10px;">Масштабные In-the-wild датасеты на разных языках. Высокая вариативность.</td>
    <td style="padding: 10px;">2025</td>
  </tr>
  <tr style="border-bottom: 1px solid #ddd">
    <td style="padding: 10px;"><b>SpeechCraft / Parler-TTS</b></td>
    <td style="padding: 10px;">2,000 / 50,000</td>
    <td style="padding: 10px;"><b>Description-based:</b> Аудио сопровождается текстовым описанием ("A man speaks calmly...").</td>
    <td style="padding: 10px;">2024</td>
  </tr>
</table>

### `Примеры метрик для современных моделей`

Оценка моделей в генерации и в способности модели скопировать голос по 3-секундному референсу на бенчмарке LibriSpeech test-clean. Больше метрик можно [посмотреть здесь](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512#evaluation).

| Модель | Архитектура | WER $\downarrow$<br>*(Один диктор / Zero-Shot)* | SS (Сходство голоса) $\uparrow$ | RTF $\downarrow$ |
| :--- | :--- | :---: | :---: | :---: |
| *Ground Truth* | *Human (LibriSpeech)* | *1.8 - 2.1* | *0.730* | *-* |
| [Tacotron 1](https://arxiv.org/abs/1703.10135) (2017) | RNN (Seq2Seq) + Griffin-Lim | \~4.0 / -  | низкое | \~0.50 - 1.0 |
| [Tacotron 2](https://arxiv.org/abs/1712.05884) (2017) | CNN + BiLSTM + Attention | \~2.5 / **>10.0** | низкое | > 1.0 (WaveNet)<br>\~0.10 (WaveGlow) |
| [FastSpeech 1](https://arxiv.org/abs/1905.09263) (2019) | Transformer (NAR) | \~3.0 / **>8.0** | низкое | ~0.03 |
|[FastSpeech 2](https://arxiv.org/abs/2006.04558) (2020) | Transformer (NAR) + Variance Adaptors| \~2.5 / **\~7.7** | низкое | **\~0.01 - 0.02** |
| [VALL-E](https://arxiv.org/abs/2301.02111) (2023) | AR + NAR | 5.9 | 0.580 | 0.50 |
| [VoiceBox](https://arxiv.org/abs/2306.15687) (2023) | Flow Matching | 1.9 | 0.681 | 0.64 |
| [CosyVoice](https://arxiv.org/abs/2407.05407) (2024)| LLM + Flow Matching | 3.2 | 0.695 | 0.92 |
| [F5-TTS](https://arxiv.org/abs/2410.06885) (2024) | Flow Matching | 2.0 | 0.647 | 0.15 |
|[Seed-TTS](https://arxiv.org/abs/2406.02430) (2024) | AR + DiT | 2.2 | **0.762** | 0.13 |